# Cobertura_Declaracao — declaração de cobertura da extração (insumo da Gold)

Tarefa do Job `v-energy-grid-360`, entre `Bronze_to_Silver` e `Silver_to_gold`. Gera a declaração de cobertura `gold-cobertura-v2` do lote Silver ativo e a entrega à Gold por *task values*, em vez de um operador preencher o JSON à mão.

**O que confere, para consumo (`leituras`) e faturamento (`faturas`):**
1. A reconciliação da Bronze do lote ativo fecha (`delta` zero e gravadas + rejeitadas = linhas de dados).
2. A Silver fecha com a Bronze (gravadas = aceitas + quarentena da Silver).
3. A Gold conta todos os registros aceitos pela Silver na janela (nenhum excluído por UC nula/de teste ou sem competência).
4. Todos os meses da janela têm registros em cada domínio.
5. Identidade da declaração: origem = aceitos + excluídos + quarentena.

Se qualquer conferência falhar, a tarefa **falha** e o Job para antes da Gold: não se declara cobertura que os números não sustentam. `registros_excluidos` é só o que a origem deixou de entregar; rejeitados por qualidade (Bronze e quarentena da Silver) entram em `registros_quarentena`. A data de corte é a data da carga na Bronze; o hash é o SHA-256 dos hashes dos arquivos de origem. Nenhum valor de negócio é impresso.

**Saídas (task values):** `declaracao_json`, `cobertura_inicio`, `cobertura_fim`, `pv_inicio_p0`, `pv_fim_p0`, `pv_inicio_p1`, `pv_fim_p1`. A janela vem das competências observadas; as janelas de preço × volume são os 12 primeiros e os 12 últimos meses, e ficam vazias se a janela tiver menos de 24 meses.

In [ ]:
import json, hashlib
from datetime import timezone
from pyspark.sql import functions as F

CAT = "workspace"

dbutils.widgets.text("DRY_RUN", "false", "true = so conferir, sem entregar a declaracao a Gold")
DRY_RUN = dbutils.widgets.get("DRY_RUN").strip().lower() == "true"
print("INFO modo:", "CONFERENCIA (nada e entregue a Gold)" if DRY_RUN else "ENTREGA")

def chk(nome, ok):
    print(("PASS " if ok else "FAIL ") + nome)
    return bool(ok)

pub = spark.table(f"{CAT}.silver.controle_publicacao").filter(F.col("chave") == "silver").collect()[0].asDict()
run_silver, lote = pub["run_ativo"], pub["lote_ativo"]
print("Silver:", run_silver, "| lote Bronze:", lote)

rec_all = spark.table(f"{CAT}.bronze_hist.reconciliacao")
col_lote = "_lote_id" if rec_all.filter(F.col("_lote_id") == lote).limit(1).count() else "_lote_origem"
rec_lote = rec_all.filter(F.col(col_lote) == lote)
quar = spark.table(f"{CAT}.silver_hist.quarentena").filter(F.col("_run_id") == run_silver)

# obrigatorio=True: falha derruba a tarefa (comportamento original).
# obrigatorio=False: falha NAO derruba; o dominio fica sem declaracao (NAO_CERTIFICADA), como hoje.
DOMINIOS = [
    {"dominio": "consumo", "tabela": "leituras", "obrigatorio": True},
    {"dominio": "faturamento", "tabela": "faturas", "obrigatorio": True},
    {"dominio": "atendimentos", "tabela": "atendimentos", "obrigatorio": False,
     "tempo": "data_abertura", "flags": ["_abertura_futura"]},
    {"dominio": "ocorrencias", "tabela": "ocorrencias_rede", "obrigatorio": False,
     "tempo": "data_abertura", "flags": ["_abertura_futura"], "exige_false": "_id_uc_ausente"},
]
uc_valida = lambda df: df.filter(F.col("id_uc").isNotNull() & ~F.col("id_uc").startswith("TESTE"))

def fonte_datada(cfg):
    """Espelha a regra da Gold (cobertura_dominios) para dominios datados por data_abertura."""
    s = uc_valida(spark.table(f"{CAT}.silver_publicada.{cfg['tabela']}"))
    if cfg.get("exige_false"):
        s = s.filter(F.col(cfg["exige_false"]) == False)  # noqa: E712
    valido = F.col("data_abertura").isNotNull() & (F.col("data_abertura") <= F.current_timestamp())
    for f in cfg["flags"]:
        valido = valido & ~F.coalesce(F.col(f), F.lit(False))
    return s.withColumn("_comp", F.when(valido, F.date_format("data_abertura", "yyyy-MM")))

# Janela: competencias observadas SOMENTE em consumo e faturamento (inalterado: as janelas de preco x volume dependem dela)
limites = []
for cfg in DOMINIOS[:2]:
    d = uc_valida(spark.table(f"{CAT}.silver_publicada.{cfg['tabela']}")).filter(F.col("mes_referencia").isNotNull())
    limites.append(d.agg(F.min("mes_referencia"), F.max("mes_referencia")).first())
INICIO = min(x[0] for x in limites)
FIM = max(x[1] for x in limites)

def indice(c): return int(c[:4]) * 12 + int(c[5:7]) - 1
def competencia(i): return f"{i // 12:04d}-{i % 12 + 1:02d}"
n_meses = indice(FIM) - indice(INICIO) + 1
print("janela de cobertura:", INICIO, "a", FIM, f"({n_meses} meses)")

declaracoes, tudo_ok, puladas = [], True, []
for cfg in DOMINIOS:
    dominio, tabela, obrigatorio = cfg["dominio"], cfg["tabela"], cfg["obrigatorio"]
    print(f"--- {dominio} ({tabela})" + ("" if obrigatorio else " [novo, nao derruba o Job]"))
    r = rec_lote.filter(F.lower(F.col("_tabela")) == tabela)
    if not chk("reconciliação Bronze encontrada para o lote", r.limit(1).count() == 1):
        if obrigatorio: tudo_ok = False
        else: puladas.append(dominio)
        continue
    agg = r.agg(F.sum("linhas_dados").alias("origem"), F.sum("gravadas").alias("gravadas"),
                F.sum("rejeitadas").alias("rejeitadas"), F.sum("delta").alias("delta"),
                F.max("_data_carga").alias("carga")).first()
    origem, gravadas, rej_bronze = int(agg["origem"]), int(agg["gravadas"]), int(agg["rejeitadas"] or 0)
    silver = spark.table(f"{CAT}.silver_publicada.{tabela}")
    silver_total = silver.count()
    q_silver = quar.filter(F.col("_tabela_origem") == tabela).count()
    if "tempo" in cfg:
        fonte = fonte_datada(cfg)
        sem_comp = fonte.filter(F.col("_comp").isNull()).count()
        no_gold = fonte.filter(F.col("_comp").between(INICIO, FIM))
        meses_com_registro = no_gold.select("_comp").distinct().count()
    else:
        sem_comp = 0
        no_gold = uc_valida(silver).filter(F.col("mes_referencia").between(INICIO, FIM))
        meses_com_registro = no_gold.select("mes_referencia").distinct().count()
    aceitos = no_gold.count()
    excluidos = silver_total - aceitos
    quarentena = rej_bronze + q_silver
    corte = agg["carga"].replace(tzinfo=timezone.utc)
    ok = chk("Bronze: delta zero em todos os arquivos", int(agg["delta"] or 0) == 0)
    ok &= chk("Bronze: gravadas + rejeitadas = linhas de dados", gravadas + rej_bronze == origem)
    ok &= chk("Silver: gravadas na Bronze = aceitos + quarentena da Silver", gravadas == silver_total + q_silver)
    ok &= chk("Gold conta todos os registros aceitos pela Silver (excluídos = 0)", excluidos == 0)
    ok &= chk("nenhum registro elegível sem competência", sem_comp == 0)
    ok &= chk("todos os meses da janela têm registros", meses_com_registro == n_meses)
    ok &= chk("identidade: origem = aceitos + excluídos + quarentena", origem == aceitos + excluidos + quarentena)
    ok &= chk("corte posterior ao início do mês seguinte à janela", corte.isoformat() >= f"{competencia(indice(FIM) + 1)}-01T00:00:00+00:00")
    if not ok:
        if obrigatorio:
            tudo_ok = False
        else:
            puladas.append(dominio)
            print(f"INFO {dominio}: não será declarado (fica NAO_CERTIFICADA, como hoje). Job segue.")
        continue
    arquivos = sorted(f"{x['_arquivo_origem']}|{x['_aba_origem'] or ''}|{x['hash_sha256']}" for x in r.collect())
    inventario = hashlib.sha256("\n".join(arquivos).encode("utf-8")).hexdigest()
    declaracoes.append({
        "dominio": dominio, "recorte": "DOMINIO", "inicio": INICIO, "fim": FIM, "estado": "FECHADO",
        "data_corte": corte.isoformat(), "referencia_evidencia": f"bronze_hist.reconciliacao/{lote}/{tabela}",
        "natureza_certificacao": "EXTRACAO",
        "reconciliacao_extracao": {"registros_origem": origem, "registros_aceitos": aceitos,
            "registros_excluidos": excluidos, "registros_quarentena": quarentena, "hash_inventario": inventario}})

n_obrig = sum(1 for c in DOMINIOS if c["obrigatorio"])
if not tudo_ok or sum(1 for c in DOMINIOS if c["obrigatorio"] and c["dominio"] in [d["dominio"] for d in declaracoes]) != n_obrig:
    raise RuntimeError("Cobertura não declarada: as conferências acima não fecham. Corrija a origem antes da Gold.")

pacote = {"schema_version": "gold-cobertura-v2", "run_id_silver": run_silver, "lote_bronze_silver": lote, "declaracoes": declaracoes}
texto = json.dumps(pacote, ensure_ascii=False, separators=(",", ":"))
assert len(texto) < 40000, "declaração excede o limite de task value"
print("INFO domínios que seriam/serão declarados FECHADO:", ", ".join(d["dominio"] for d in declaracoes))
print("INFO domínios que ficam NAO_CERTIFICADA:", ", ".join(puladas) if puladas else "nenhum dos novos")

# Janelas de preço x volume: 12 primeiros e 12 ultimos meses (vazias se houver menos de 24 meses)
if n_meses >= 24:
    pv = (INICIO, competencia(indice(INICIO) + 11), competencia(indice(FIM) - 11), FIM)
else:
    pv = ("", "", "", "")
valores = {"declaracao_json": texto, "cobertura_inicio": INICIO, "cobertura_fim": FIM,
           "pv_inicio_p0": pv[0], "pv_fim_p0": pv[1], "pv_inicio_p1": pv[2], "pv_fim_p1": pv[3]}
if DRY_RUN:
    print("PASS conferência concluída; nenhuma declaração foi entregue (DRY_RUN)")
else:
    for chave, valor in valores.items():
        dbutils.jobs.taskValues.set(key=chave, value=valor)
    print("PASS declaração gerada e entregue à Gold (tamanho em caracteres):", len(texto))
print("janelas de preço x volume:", pv)
